# Hello, Symbolic-KAN Reproducible

> **Unofficial derivative tutorial.** Cite the original Symbolic-KAN paper and repository. Outputs from this notebook are derivative-package results, not values supplied by the upstream authors.

This notebook fits a small analytic target, inspects pre-hardening primitive evidence, saves versioned checkpoints, and exports auditable reports.

In [1]:
from pathlib import Path
import torch

from symbolic_kan import (
    ModelConfig, SymbolicKAN, TrainingConfig, fit_supervised,
    rank_primitive_candidates, save_checkpoint, write_symbolic_report,
)

torch.manual_seed(42)

## Target

We use $y=\exp(\sin(\pi x_0)+x_1^2)$ because its compact analytic structure makes the exported candidate evidence easy to inspect.

In [2]:
dtype = torch.float64
x = 2 * torch.rand(384, 2, dtype=dtype) - 1
y = torch.exp(torch.sin(torch.pi * x[:, [0]]) + x[:, [1]].square())
x_train, x_val = x[:256], x[256:]
y_train, y_val = y[:256], y[256:]

In [3]:
model = SymbolicKAN(ModelConfig(
    input_dim=2, hidden_units=4, edges_per_unit=2, num_blocks=2,
    primitives=("x", "x2", "sin", "cos", "exp"),
    readout="fixed_sum", dtype="float64",
))
training = TrainingConfig(
    seed=42, adam_epochs=100, lbfgs_steps=2,
    deterministic_algorithms=False,
)
result = fit_supervised(
    model, x_train, y_train, x_val, y_val, training,
    output_directory="outputs/hello-symbolic-kan",
)

## Inspect and export

Load the best soft state before ranking. Scores combine deterministic gate probability with a documented complexity prior; they are not $R^2$ values and do not certify the true governing law.

In [4]:
soft_model = SymbolicKAN(model.config)
soft_state_path = Path("outputs/hello-symbolic-kan/model_best_soft.pt")
try:
    soft_state = torch.load(soft_state_path, map_location="cpu", weights_only=True)
except TypeError:
    soft_state = torch.load(soft_state_path, map_location="cpu")
soft_model.load_state_dict(soft_state)
soft_model.eval()
candidates = rank_primitive_candidates(soft_model, top_k=3)
[candidate.to_dict() for candidate in candidates[:6]]

[{'block': 0,
  'unit': 0,
  'edge': 0,
  'rank': 1,
  'primitive': 'x',
  'probability': 0.20002184732036687,
  'complexity': 1.0,
  'score': 0.18002184732036688},
 {'block': 0,
  'unit': 0,
  'edge': 0,
  'rank': 2,
  'primitive': 'x2',
  'probability': 0.20084904171395263,
  'complexity': 1.5,
  'score': 0.17084904171395263},
 {'block': 0,
  'unit': 0,
  'edge': 0,
  'rank': 3,
  'primitive': 'sin',
  'probability': 0.20083651373195835,
  'complexity': 2.0,
  'score': 0.16083651373195834},
 {'block': 0,
  'unit': 0,
  'edge': 1,
  'rank': 1,
  'primitive': 'x',
  'probability': 0.20063075356292728,
  'complexity': 1.0,
  'score': 0.1806307535629273},
 {'block': 0,
  'unit': 0,
  'edge': 1,
  'rank': 2,
  'primitive': 'x2',
  'probability': 0.2006558545080666,
  'complexity': 1.5,
  'score': 0.1706558545080666},
 {'block': 0,
  'unit': 0,
  'edge': 1,
  'rank': 3,
  'primitive': 'cos',
  'probability': 0.20087526593002086,
  'complexity': 2.0,
  'score': 0.16087526593002086}]

In [5]:
output = Path("outputs/hello-symbolic-kan")
save_checkpoint(output / "checkpoint_soft.pt", soft_model, phase="soft", training_config=training, history=result.history, metadata={"profile": "tutorial", "seed": 42})
soft_paths = write_symbolic_report(soft_model, output / "soft_report", variables=["x_0", "x_1"], history=result.history, metadata={"profile": "tutorial", "seed": 42})
save_checkpoint(output / "checkpoint_hardened.pt", result.model, phase="hardened", training_config=training, history=result.history, metadata={"profile": "tutorial", "seed": 42})
hardened_paths = write_symbolic_report(result.model, output / "hardened_report", variables=["x_0", "x_1"], history=result.history, metadata={"profile": "tutorial", "seed": 42})
{"soft": soft_paths, "hardened": hardened_paths}

{'soft': {'report': WindowsPath('outputs/hello-symbolic-kan/soft_report/symbolic_report.json'),
  'expression': WindowsPath('outputs/hello-symbolic-kan/soft_report/expression.txt'),
  'structure': WindowsPath('outputs/hello-symbolic-kan/soft_report/structure.svg'),
  'html': WindowsPath('outputs/hello-symbolic-kan/soft_report/report.html')},
 'hardened': {'report': WindowsPath('outputs/hello-symbolic-kan/hardened_report/symbolic_report.json'),
  'expression': WindowsPath('outputs/hello-symbolic-kan/hardened_report/expression.txt'),
  'structure': WindowsPath('outputs/hello-symbolic-kan/hardened_report/structure.svg'),
  'html': WindowsPath('outputs/hello-symbolic-kan/hardened_report/report.html')}}

Open `outputs/hello-symbolic-kan/soft_report/report.html` to inspect pre-hardening candidate evidence and `hardened_report/report.html` for the exported discrete result. For a scientific claim, repeat across seeds and test derivatives, extrapolation, and dimensional consistency.